In [ ]:


export=False


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
from datetime import datetime
import time
import functools as ft
from IPython.display import display
pd.set_option('display.max_columns', None)


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_csm = path_prod / 'Chamber Study Missions' / date.today().strftime('%Y') / '2026 peer region tour'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())



In [ ]:


file_pop3 = path_csm / f'Pop_3 MSA ACS5_ChamberStudy2026.xlsx'
file_pop4 = path_csm / f'Pop_4 MSA ACS5_ChamberStudy2026.xlsx'
file_edu1 = path_csm / f'Edu_1 MSA ACS5_ChamberStudy2026.xlsx'


df_pop3 = pd.read_excel(file_pop3, sheet_name='MSA')
df_pop4 = pd.read_excel(file_pop4, sheet_name='MSA')
df_edu1 = pd.read_excel(file_edu1, sheet_name='MSA')



QC

In [ ]:
# Set Indicator
indicator = 'Pop_3'
plot_name = 'race_ethnicity'


file_name = path_csm / f'{indicator} MSA ACS5_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')
df_msa['MSA'] = df_msa['MSA'].map(peer_msa_labels)


df_msa['Percentage'] = round(df_msa['Percentage']*100, 1)
df_msa = df_msa[df_msa['Race_Ethnicity'] != 'All']


df_msa['Sort'] = pd.Categorical(df_msa['Race_Ethnicity'], [
    'American Indian or Alaska Native (NH)'
    , 'Native Hawaiian or other Pacific Islander (NH)'
    , 'Some other race (NH)'
    , 'Two or more races (NH)'
    , 'Black or African American (NH)'
    , 'Asian (NH)'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_msa = df_msa.sort_values('Sort', ascending=False)
df_msa = df_msa.drop(['Sort'], axis=1)
df_msa = df_msa.reset_index(drop=True)

display(df_msa.head())



## Plotting ---

color_map  = {
    'American Indian or Alaska Native (NH)': '#A97142'
    , 'Native Hawaiian or other Pacific Islander (NH)': '#006A4E'
    , 'Some other race (NH)': '#7E587E'
    , 'Two or more races (NH)': '#1F45FC'
    , 'Asian (NH)': '#9DC209'
    , 'Black or African American (NH)': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.bar(df_msa, x='Year', y='Percentage', color='Race_Ethnicity', color_discrete_map=color_map, facet_row='MSA')


title = '<b>Race and Ethnicity</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

fig.for_each_yaxis(lambda x: x.update(title=''))
fig.for_each_annotation(lambda a: a.update(text=re.sub('name=', '', a.text)))
for annotation in fig['layout']['annotations']: 
    annotation['textangle']=0


plot_agol(export=export)


In [ ]:


# Set Indicator
indicator = 'Pop_4'
plot_name = 'age_groups'
geography = 'MSA'



file_name = path_csm / f'{indicator} MSA ACS5_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')
df_msa['MSA'] = df_msa['MSA'].map(peer_msa_labels)



## Organizing ---

df_msa['Percentage'] = round(df_msa['Percentage']*100, 1)
df_msa = df_msa[df_msa['Race_Ethnicity'] == 'All']


df_msa['Sort'] = pd.Categorical(df_msa['Variable'], [
    'Under 18'
    , '18 to 64'
    , '65+'
])
    
df_msa = df_msa.sort_values('Sort', ascending=False)
df_msa = df_msa.drop(['Sort'], axis = 1)
df_msa = df_msa.reset_index(drop = True)

display(df_msa.head())


## Plotting ---
color_map = {
    'Under 18': '#9DC209'
    , '18 to 64': '#1E90FF'
    , '65+': '#1F45FC'
}


fig = px.bar(df_msa, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map, facet_row = geography)


title = '<b>Population Age Distribution</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=20, range = [0, 105], ticksuffix='%')
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

fig.for_each_yaxis(lambda x: x.update(title=''))
fig.for_each_annotation(lambda a: a.update(text=re.sub('name=', '', a.text)))
for annotation in fig['layout']['annotations']: 
    annotation['textangle']=0


plot_agol(export=export)


In [ ]:



# Set Indicator
indicator = 'Edu_1'
plot_name = '2022'


## Importing ---

file_name = path_csm / f'{indicator} MSA ACS5_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')


## Organizing ---

for msa in df_msa.MSA.unique():

    df_plot = df_msa[df_msa['MSA'] == msa]

    df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
    df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]


    df_plot['Sort'] = pd.Categorical(df_plot['Variable'], [
        'Total Less than high school diploma'
        , 'Total High school graduate or GED'
        , "Total Some college or associate's degree"
        , "Total Bachelor's degree or higher"
    ])
        
    df_plot = df_plot.sort_values(['Sort'], ascending = [False])
    df_plot = df_plot.drop(['Sort'], axis = 1)

    df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
    df_plot = df_plot.reset_index(drop = True)

    display(df_plot.head())

    ## Plotting ---
    color_map  = {
        'Total Less than high school diploma':'#FBB117'
        , 'Total High school graduate or GED':'#9DC209'
        , "Total Some college or associate's degree":'#1E90FF'
        , "Total Bachelor's degree or higher":'#1F45FC'
    }


    fig = px.bar(df_plot, y='Percentage', x='Race_Ethnicity'
                , color='Variable'
                , color_discrete_map=color_map)


    title = f'<b>Educational Attainment by Race/Ethnicity ({df_plot.Year.max()}) {msa}</b>'
    fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
    fig.update_traces(hovertemplate='%{y}')
    fig.update_layout(legend={'traceorder': 'reversed'})


    plot_agol(export=export)


In [ ]:


# Set Indicator
indicator = 'Income_4'
plot_name = 'poverty_rate'


## Importing ---

file_name = path_csm / f'{indicator} MSA ACS5_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')


## Organizing ---


for msa in df_msa.MSA.unique():

    df_plot = df_msa[df_msa['MSA'] == msa]

    race_ethnicity = ['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)']
    df_plot = df_plot[df_plot['Race_Ethnicity'].isin(race_ethnicity)]
    df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
    df_plot = df_plot[df_plot['Variable'] == 'Total Income in the past 12 months below poverty level']
    df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
    df_plot = df_plot.reset_index(drop = True)


    display(df_plot.head())


    ## Plotting ---
    fig = px.bar(df_plot, x='Race_Ethnicity', y='Percentage')
    fig.update_traces(marker_color='#1E90FF')


    title = f'<b>Relative Poverty by Race/Ethnicity ({df_plot.Year.max()}) {msa}'
    fig.update_yaxes(dtick=5, ticksuffix='%', range = [0,27])
    fig.update_xaxes(dtick=1)
    fig.update_traces(hovertemplate="%{y}")

    plot_agol(export=export)



In [ ]:
# Set Indicator
indicator = 'Cost_5'
plot_name = 'owners'
geography = 'MPO'


## Importing ---

file_name = path_csm / f'{indicator} MSA ACS5_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')



## Organizing ---



for msa in df_msa.MSA.unique():

    df_plot = df_msa[df_msa['MSA'] == msa]

    df_plot = df_plot[df_plot['Race_Ethnicity'].isin(['Asian', 'Black or African American', 'Hispanic or Latino', 'White (NH)'])]
    df_plot = df_plot[df_plot['Variable'] == 'Owner occupied']
    df_plot['Percentage'] = round(df_plot['Percentage']*100, 1)
    display(df_plot.head())


    ## Plotting ---

    color_map  = {
        'Asian': '#9DC209'
        , 'Black or African American': '#1E90FF'
        , 'Hispanic or Latino': "#FBB117"
        , 'White (NH)': "#DC381F"
    }

    fig = px.line(df_plot, x='Year', y='Percentage', color='Race_Ethnicity', markers = True,
                color_discrete_map=color_map)

    title = f'<b>Owner Occupied Housing by Race/Ethnicity</b> ({msa})'
    fig.update_yaxes(dtick=10, ticksuffix='%', range = [0, 100])
    fig.update_xaxes(dtick=1, range=[df_plot.Year.min()-0.5, df_plot.Year.max()+0.5])
    fig.update_traces(hovertemplate="%{y}")


    plot_agol(export=export)


In [ ]:
# Set Indicator
indicator = 'Cost_6'
plot_name = 'burden'
geography = 'MSA'


## Importing ---

file_name = path_csm / f'{indicator} MSA PUMS_ChamberStudy2026.xlsx'
df_msa = pd.read_excel(file_name, sheet_name='MSA')


## Organizing ---


for msa in df_msa.MSA.unique():

    df_plot = df_msa[df_msa['MSA'] == msa]

    df_plot = df_plot[df_plot['Housing Type'].isin(['Owner', 'Renter'])]

    race_ethnicity = ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)']
    df_plot = df_plot[df_plot['RAC1P'].isin(race_ethnicity)]
    df_plot = df_plot[df_plot['Year'] == df_plot.Year.max()]
    df_plot = df_plot.groupby(['MSA', 'Year', 'RAC1P', 'Housing Burden'], as_index = False).agg(Total = ('Households', 'sum'))
    df_plot['Percentage'] = 100*df_plot['Total'] / df_plot.groupby(['MSA', 'Year', 'RAC1P'])['Total'].transform('sum')
    df_plot = df_plot[df_plot['Housing Burden'].isin(['Cost burden >30% to <=50%', 'Cost burden >50%'])]

    display(df_plot.head())


    df_plot['Housing_sort'] = pd.Categorical(df_plot['Housing Burden'], [
        'Cost burden >50%'
        , 'Cost burden >30% to <=50%'
    ])
        
    df_plot = df_plot.sort_values(['Housing_sort'], ascending = [False])
    df_plot = df_plot.drop(['Housing_sort'], axis = 1)

    df_plot['Percentage'] = round(df_plot['Percentage'], 1)


    ## Plotting ---

    color_map_cost  = {
        'Cost burden >50%': '#9DC209'
        , 'Cost burden >30% to <=50%': '#1E90FF'
    }


    fig = px.bar(df_plot, y='Percentage', x='RAC1P'
                , color='Housing Burden'
                , color_discrete_map=color_map_cost)


    title = f'<b>Housing Cost Burden by Race/Ethnicity, {df_plot.Year.max()}</b>  <br><sup>6-County Sacramento Region</sup> '
    fig.update_yaxes(tick0=0, dtick=10, ticksuffix='%')
    fig.update_traces(hovertemplate='%{y}')
    fig.update_layout(legend={'traceorder': 'reversed'})


    plot_agol(export=export)
